In [0]:
raw_path = "s3://voyage-data-plateform-yuvaraj-india/raw/airbyte/voyage_raw/"

display(dbutils.fs.ls(raw_path))

### validation of staging views created using DBT.

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM voyage.dbt_dev.stg_bookings;

In [0]:
%sql
SELECT
    booking_id,
    tenant_id,
    property_id,
    customer_id,
    booking_date,
    adr,
    status,
    updated_at
FROM voyage.dbt_dev.stg_bookings
ORDER BY booking_id
LIMIT 10;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM voyage.silver.fact_booking) AS silver_rows,

    (SELECT COUNT(*)
     FROM voyage.dbt_dev.stg_bookings) AS dbt_staging_rows;

In [0]:
%sql
SELECT count(*) AS Total_rows
FROM voyage.dbt_dev.stg_customers;

In [0]:
%sql
SELECT * FROM voyage.dbt_dev.stg_customers
LIMIT 10;

In [0]:
%sql
SELECT*FROM voyage.dbt_dev.stg_properties;

In [0]:
%sql
SELECT*FROM voyage.dbt_dev.stg_tenants;

In [0]:
%sql
SELECT
    SUM(booking_count) AS total_bookings,
    SUM(completed_booking_count) AS completed_bookings,
    SUM(confirmed_booking_count) AS confirmed_bookings,
    SUM(cancelled_booking_count) AS cancelled_bookings,
    SUM(room_nights) AS total_room_nights,
    SUM(estimated_completed_booking_value) AS estimated_completed_value
FROM voyage.silver.int_booking_daily;

In [0]:
%sql
select
    customer_id,
    tenant_id,
    count(*) as row_count
from voyage.dbt_dev.int_customer_booking_summary
group by
    customer_id,
    tenant_id
having count(*) > 1;

In [0]:
%sql
select
    count(*) as customer_summary_rows,
    count(distinct customer_id) as distinct_customers
from voyage.dbt_dev.int_customer_booking_summary;

In [0]:
%sql
DESCRIBE TABLE voyage.silver.dim_tenant;

In [0]:
%sql
DESCRIBE TABLE voyage.bronze.tenants;

In [0]:
%sql
CREATE OR REPLACE TABLE voyage.silver.dim_tenant
USING DELTA
AS

WITH ranked AS (

    SELECT
        tenant_id,
        tenant_name,
        country,
        currency,
        to_timestamp(created_at) AS created_at,
        to_timestamp(updated_at) AS updated_at,
        _airbyte_extracted_at,
        ROW_NUMBER() OVER (
            PARTITION BY tenant_id
            ORDER BY
                to_timestamp(updated_at) DESC,
                _airbyte_extracted_at DESC
        ) AS rn

    FROM voyage.bronze.tenants

)

SELECT
    tenant_id,
    tenant_name,
    country,
    currency,
    created_at,
    updated_at

FROM ranked

WHERE rn = 1;

In [0]:
%sql
DESCRIBE TABLE voyage.silver.dim_tenant;

In [0]:
%sql
DESCRIBE TABLE voyage.silver.dim_customer;

In [0]:
%sql
DESCRIBE TABLE voyage.silver.dim_property;

In [0]:
%sql
CREATE OR REPLACE TABLE voyage.silver.dim_customer
USING DELTA
AS

WITH ranked AS (

    SELECT
        customer_id,
        tenant_id,
        country,
        customer_type,
        to_timestamp(created_at) AS created_at,
        to_timestamp(updated_at) AS updated_at,
        _airbyte_extracted_at,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY
                to_timestamp(updated_at) DESC,
                _airbyte_extracted_at DESC
        ) AS rn

    FROM voyage.bronze.customers

)

SELECT
    customer_id,
    tenant_id,
    country,
    customer_type,
    created_at,
    updated_at

FROM ranked

WHERE rn = 1;

In [0]:
%sql
SELECT COUNT(*) AS customer_count
FROM voyage.silver.dim_customer;

In [0]:
%sql
SELECT COUNT(*) AS tenant_count
FROM voyage.silver.dim_tenant;